# Lab 03 — Capture the activations (the tensors the whole thesis is about)

**Goal:** intercept the B×S×H tensor between transformer blocks with **forward hooks**, measure its size (= your wire cost at a pipeline boundary), and observe the **outlier problem** on real data — the phenomenon that justifies per-block quantization scales.

In [ ]:
!pip -q install transformers matplotlib
import torch, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

## 1. Forward hooks — reading a tensor mid-flight
A hook is a function PyTorch calls with `(module, input, output)` every time that module runs. It is the single most useful research tool in this project: capture, measure, or *modify* any intermediate tensor without touching model code.

In [ ]:
acts = {}
def grab(name):
    def fn(module, inp, out):
        acts[name] = (out[0] if isinstance(out, tuple) else out).detach()
    return fn

hooks = [blk.register_forward_hook(grab(f"block{i}")) for i, blk in enumerate(model.transformer.h)]

long_ids = tok(" systems" * 512, return_tensors="pt").to(device)   # S ~ 512
with torch.no_grad():
    model(**long_ids)
for h in hooks: h.remove()

a = acts["block5"]
print("shape (B,S,H):", a.shape, a.dtype)

## 2. Wire cost — what this tensor costs on your network
This is the exact quantity `S · d / B_i` from the proposal's cost model.

In [ ]:
bytes_fp16 = a.numel() * 2
for gbps in [1, 10]:
    eff = gbps * 1e9 / 8 * 0.94          # ~6% TCP/framing overhead
    print(f"{bytes_fp16/1e6:6.2f} MB  over {gbps:2d} Gbps: {bytes_fp16/eff*1000:7.2f} ms   (4-bit: {bytes_fp16/4/eff*1000:6.2f} ms)")
print("PCIe 4.0 x16 (~25 GB/s eff):", f"{bytes_fp16/25e9*1000:.3f} ms")

That gap — milliseconds on PCIe vs ~a hundred+ on Ethernet — is the Communication Wall, measured by you.

## 3. The outlier problem, live
Compare the max absolute value to the 99.9th percentile per layer. A large ratio means a handful of values dominate the range — exactly what wrecks a shared quantization scale.

In [ ]:
for name in [f"block{i}" for i in range(0, 12, 2)]:
    t = acts[name].flatten().float()
    q = t.abs().quantile(0.999)
    print(f"{name:8s} max|x|={t.abs().max():8.1f}   p99.9={q:6.1f}   ratio={t.abs().max()/q:5.1f}x")

In [ ]:
t = acts["block10"].flatten().float().cpu()
plt.figure(figsize=(7,3))
plt.hist(t, bins=300, log=True)
plt.title("Activation values, block 10 (log count) — note the heavy tails")
plt.xlabel("value"); plt.ylabel("count (log)"); plt.tight_layout(); plt.show()

Heavy tails on a log histogram = the outlier channels the literature (and TAH-Quant's Hadamard trick) exists for. **Save this figure — it's supervisor-meeting evidence.**

## Exercises
1. Vary S (128, 512, 1024) and plot boundary-tensor MB vs S — confirm communication scales with sequence length (the low-resource-language tokenizer argument).
2. Plot the max/p99.9 ratio for **all** 12 layers. Which layers are most outlier-prone? (These will be the hardest to quantize in Lab 04.)
3. Hook `inp` instead of `out` and confirm block i's input == block i-1's output.